# CNN Experiments

- Run the CNN architectures, ResNet-50 andtrained with and without background randomisation augmentation, across multiple seeds.

# Importing libraries

In [1]:
import pathlib
import json
import torch
import numpy as np
import random
import pandas as pd
import time

from cory_data_wrapper import *
from cory_utils import *

# Select device

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device}")

Using cuda


# Create constants and create paths

In [3]:
# Make sure that the directory exists
IMAGE_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 10
SEEDS = [0, 339, 1]
CNN_ARCHITECTURES = ["resnet50"]

DATA_ROUTE = r"E:\AIML339\in9L_annotated_only"
BOUNDING_BOX_PATH = "LOC_train_solution.csv"
TEST_INSTANCES = pathlib.Path(r"E:\AIML339\backgrounds_challenge_data\bg_challenge")
RESULTS = pathlib.Path("./results")
TENSORBOARD_DIR = pathlib.Path("./runs")
RESULTS.mkdir(exist_ok=True)
TENSORBOARD_DIR.mkdir(exist_ok=True)

# Load the data

In [4]:
# Load data
bboxs = load_bbox_csv(BOUNDING_BOX_PATH)
base_train_dataset = IN9BoundingBoxDataset(DATA_ROUTE, bboxs, image_size=IMAGE_SIZE)
train_class_to_idx = base_train_dataset.class_to_idx


# print the instances and num of classes
print(f"Loaded {len(base_train_dataset)} images", "with" , len(base_train_dataset.classes), "classes")

Loaded 69120 images with 9 classes


# Split the data (Train, Validation)

In [5]:
train_subset, val_subset = stratified_train_val_split(base_train_dataset, val_fraction=0.1, seed=0)

# Create data loaders for all sets, for faster loading and shuffling and batches
# Create the validation loader
val_dataset_normalised = BackgroundRandomizationDataset(val_subset, augmentation_strength=0.0, seed=0)
val_loader = make_loader(val_dataset_normalised, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# make loaders for the test set, one for each variant
test_variant_loaders = {
    variant: make_loader(
        fix_test_labels(
            ImageFolderEval(TEST_INSTANCES / variant / "val", image_size=IMAGE_SIZE),train_class_to_idx
        ),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=4
    )
    for variant in ["original", "no_fg", "mixed_same", "mixed_rand"]
}
print("Test variants loaders created:", list(test_variant_loaders.keys()))

Test variants loaders created: ['original', 'no_fg', 'mixed_same', 'mixed_rand']


# Perform Learning Rate Search

- Perform a small learning rate search on Validation Accuracy
- Train for 3 epochs on 3 different learning rates.

In [6]:
# Create the dictionary for best Learning-Rate
BEST_LR = {
    "resnet50": 3e-05
}
# added the value so it didnt run a second time
best_lr_per_arch = {
    "resnet50": 3e-05}
# loop through all architectures and perform the Hyper Parameter search for each architecture
for arch_name in CNN_ARCHITECTURES:
    # Check to ensure that it only runs if LR not known,
    if BEST_LR[arch_name] is not None:
        best_lr_per_arch[arch_name] = BEST_LR[arch_name]
        continue

    print(f"\n{'-'*60}\n Hyper Parameter Search: {arch_name} \n{'-'*60}")
    #
    search_train_loader = make_loader(train_subset,batch_size=BATCH_SIZE, shuffle=True, num_workers=4)

    best_lr, hyperparameter_trials = hyperparameter_search(
        lambda a = arch_name: CNNModel(a, num_classes=9, pretrained=True),
        search_train_loader, val_loader, device,
        learning_rates=(1e-5, 3e-5, 1e-4), search_epochs=3,
        log_dir=TENSORBOARD_DIR / "hpsearch" / arch_name
    )
    best_lr_per_arch[arch_name] = best_lr
    print(f"{arch_name} best LR: {best_lr:}")

print("\nBest LR per architecture:", best_lr_per_arch)


Best LR per architecture: {'resnet50': 3e-05}


# Architecture Train, Eval and Test Function

In [7]:
def run_cnn_experiment(arch_name, augmented, seed):
    """
    Run the CNN experiment for a specified architecture and seed.
    Stores the data in results folder

    @param arch_name: Name of the CNN architecture.
    @param augmented: Whether to use augmented or original data.
    @param seed: Seed for reproducibility.
    @return: A dictionary containing the results of the specified experiment.
    """
    data_type = "augmented" if augmented else "original"
    run_name = f"{arch_name}_{data_type}_{seed}"
    print(f"\n{'='*60}\n{run_name}\n{'='*60}")

    # If data is augmented, strength = 1 (swap) else 0 (do not swap background), swap the background with a random image's background
    train_dataset_for_run = BackgroundRandomizationDataset(
        train_subset, augmentation_strength=1.0 if augmented else 0.0, seed=seed
    )
    train_loader = make_loader(train_dataset_for_run, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)

    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    # Define the model and move it to the device
    model = CNNModel(arch_name, num_classes=9, pretrained=True).to(device)

    start_time = time.time()
    # Train the model and store the results
    trained_model, history = train_model(
        model, train_loader, val_loader, device, epochs=EPOCHS, early_stopping_patience=3, model_name=run_name, lr=best_lr_per_arch[arch_name], log_dir=TENSORBOARD_DIR / run_name
    )
    training_time_in_secs = time.time() - start_time

    # Save the model weights
    weights_path = RESULTS / f"{run_name}_weights.pt"
    torch.save(trained_model.state_dict(), weights_path)

    variant_accuracies = evaluate_on_variants(trained_model, test_variant_loaders, device)
    # compute the reliance metrics
    reliance = compute_reliance_metrics(variant_accuracies, num_classes=9)
    print(f"Test variants: {variant_accuracies}")
    print(f"BG-Gap: {reliance['bg_gap']:.4f}, No-FG: {reliance['no_fg_accuracy']:.4f}, Original: {reliance['original_accuracy']:.4f}")
    # Store the results in a dictionary
    results = {
        "architecture" : arch_name,
        "data_type" : data_type,
        "seed" : seed,
        "variant_accuracies" : variant_accuracies,
        "reliance_metrics" : reliance,
        "final_val_accuracy" : history["val_accuracy"][-1],
        "epochs_run" : len(history["val_accuracy"]),
        "training_time_in_secs" : training_time_in_secs
    }
    # Save the dadta
    results_path = RESULTS / "cnn_results.json"
    results_each_run = {}
    if results_path.exists():
        with open(results_path, "r") as f:
            for r in json.load(f):
                key = f"{r['architecture']}_{r['data_type']}_{r['seed']}"
                results_each_run[key] = r
    results_each_run[run_name] = results
    with open(results_path, "w") as f:
        json.dump(list(results_each_run.values()), f, indent=2)

    return results

# Run Each model and Seed

### ResNet50 (No Augmentation)

In [8]:
run_cnn_experiment("resnet50", augmented=False, seed=0)


resnet50_original_0
[resnet50_original_0] Epoch 1/10: train_loss=0.7798  train_acc=0.7800 val_loss=0.0493  val_acc=0.9868
[resnet50_original_0] Epoch 2/10: train_loss=0.0590  train_acc=0.9828 val_loss=0.0379  val_acc=0.9896
[resnet50_original_0] Epoch 3/10: train_loss=0.0192  train_acc=0.9946 val_loss=0.0437  val_acc=0.9903
[resnet50_original_0] Epoch 4/10: train_loss=0.0099  train_acc=0.9972 val_loss=0.0480  val_acc=0.9887
[resnet50_original_0] Epoch 5/10: train_loss=0.0052  train_acc=0.9983 val_loss=0.0551  val_acc=0.9897
[resnet50_original_0] Epoch 6/10: train_loss=0.0033  train_acc=0.9990 val_loss=0.0469  val_acc=0.9910
[resnet50_original_0] Epoch 7/10: train_loss=0.0019  train_acc=0.9995 val_loss=0.0493  val_acc=0.9910
[resnet50_original_0] Epoch 8/10: train_loss=0.0016  train_acc=0.9995 val_loss=0.0483  val_acc=0.9907
[resnet50_original_0] Epoch 9/10: train_loss=0.0005  train_acc=0.9999 val_loss=0.0504  val_acc=0.9916
[resnet50_original_0] Epoch 10/10: train_loss=0.0004  train_a

{'architecture': 'resnet50',
 'data_type': 'original',
 'seed': 0,
 'variant_accuracies': {'original': 0.9869135802469136,
  'no_fg': 0.5995061728395061,
  'mixed_same': 0.9530864197530864,
  'mixed_rand': 0.9125925925925926},
 'reliance_metrics': {'chance_accuracy': 0.1111111111111111,
  'no_fg_accuracy': 0.5995061728395061,
  'bg_gap': 0.04049382716049377,
  'original_accuracy': 0.9869135802469136},
 'final_val_accuracy': 0.9917534722222222,
 'epochs_run': 10}

In [ ]:
run_cnn_experiment("resnet50", augmented=False, seed=339)


resnet50_original_339
[resnet50_original_339] Epoch 1/10: train_loss=0.7868  train_acc=0.7748 val_loss=0.0493  val_acc=0.9878


In [ ]:
run_cnn_experiment("resnet50", augmented =False, seed = 1)

### ResNet50 (Augmentation)


In [ ]:
run_cnn_experiment("resnet50", augmented=True, seed=0)

In [ ]:
run_cnn_experiment("resnet50", augmented=True, seed=339)

In [ ]:
run_cnn_experiment("resnet50", augmented=True, seed = 1)

# Short Results Summary

In [ ]:
results_path = RESULTS / "cnn_results.json"
with open(results_path, "r") as f:
    results = json.load(f)

rows = []
for r in results:
    row = {"architecture": r["architecture"], "data_type": r["data_type"], "seed": r["seed"], "final_val_accuracy": r["final_val_accuracy"], "epochs_run": r["epochs_run"]}
    row.update(r["reliance_metrics"])
    row["original_accuracy"] = r["variant_accuracies"].get("original")
    rows.append(row)

df = pd.DataFrame(rows)
summary = df.groupby(["architecture", "data_type"]).agg(
    mean_bg_gap =("bg_gap", "mean"), std_bg_gap =("bg_gap", "std"),
    mean_no_fg =("no_fg_accuracy", "mean"), std_no_fg =("no_fg_accuracy", "std"),
    mean_original =("original_accuracy", "mean"), std_original =("original_accuracy", "std")
)
summary

# Show TensorBoard

In [ ]:
# %load_ext tensorboard
# %tensorboard --logdir ./runs

# GradCam

In [ ]:
from cory_gradcam_ import get_resnet50_gradcam
from PIL import Image
import pathlib
import torch

model = CNNModel("resnet50", num_classes=9, pretrained=False)
# Print out the parameter count
print(f"ResNet50 Parameter Count: {sum(p.numel() for p in model.parameters()):,}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RESULTS = pathlib.Path("./results")

bboxs = load_bbox_csv(r"LOC_train_solution.csv")
base_data = IN9BoundingBoxDataset(r"E:\AIML339\in9L_annotated_only", bboxs, image_size=224)
eval_ds = BackgroundRandomizationDataset(base_data, augmentation_strength=0.0, seed=0)

def perform_gradcam(augmented=False, img_index = 339):
    if augmented:
        model.load_state_dict(torch.load(RESULTS / f"resnet50_augmented_0_weights.pt"))
    else:
        model.load_state_dict(torch.load(RESULTS / f"resnet50_original_0_weights.pt"))
    model.to(device)
    image, label = eval_ds[img_index]

    overlay, gradcam, predicted = get_resnet50_gradcam(model, image, label, device)
    print(f"True label: {label}, Predicted label: {predicted}")
    Image.fromarray(overlay).save(f"resnet50_gradcam_{'augmented' if augmented else 'original'}_{img_index}.png")
    Image.fromarray((gradcam * 255).astype(np.uint8)).save(f"resnet50_gradcam_{'augmented' if augmented else 'original'}heatmap_{img_index}.png")

perform_gradcam(augmented=False, img_index=339)
perform_gradcam(augmented=True, img_index=339)